# T20 / E02 — Tái lập Lookback Lens gốc

Đây là **mốc so sánh nội bộ quan trọng nhất** của đề tài, vì đóng góp chính là một sửa đổi trực
tiếp trên phương pháp này. Nếu `chunk-aware` chỉ vượt được một bản tái lập **yếu** thì phép so
sánh không chứng minh được gì.

Công thức gốc và bốn điểm dễ làm sai nằm ở mục 1 `docs/REFERENCES.md`.

## Đây là lượt GPU nặng nhất từ đầu dự án

Khác mọi lượt trước, lần này chạy mô hình đọc **Qwen2.5-7B lượng tử hóa 4 bit** dưới chế độ
teacher forcing để lấy ma trận chú ý, chứ không tinh chỉnh gì.

| Việc | Số mẫu | Ước tính |
|---|---|---|
| Trích đặc trưng tập train | 5.600 | ~39 phút |
| Trích đặc trưng tập test | 700 | ~5 phút |
| Huấn luyện và chấm điểm | — | vài giây, chạy CPU |

Đo ở T08: khoảng **1,05 ms mỗi token prompt**, ra chừng **420 ms mỗi mẫu** trên ViHallu. Cộng
thời gian tải mô hình 7B thì tổng khoảng **50 phút**.

## Chạy dở vẫn nối lại được

Mỗi mẫu được ghi xuống **ngay khi tính xong**, và chạy lại sẽ bỏ qua phần đã có. Nghĩa là một
lượt bị đứt phiên, hết hạn mức hay bấm nhầm Ctrl+C vẫn giữ nguyên phần đã trả tiền — chạy lại
đúng lệnh cũ là đi tiếp từ chỗ dừng. Cùng lý do với cache của T19.

## Notebook settings

- Accelerator: **GPU T4 x2**
- Internet: **On**
- Data: attach dataset `unicorn1209/vihallulens`

## Chuẩn bị

Ba ô dưới đây giống notebook T18. Khoảng 2 phút.

In [ ]:
# Ô 1 — lấy code. Chạy lại được nhiều lần.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    done = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(" ".join(args) + chr(10) + done.stdout + done.stderr)
    return done.stdout.strip()


if (REPO_DIR / ".git").is_dir():
    run("git", "fetch", "--quiet", "origin", cwd=REPO_DIR)
    run("git", "reset", "--quiet", "--hard", "origin/main", cwd=REPO_DIR)
    print("đã cập nhật repo có sẵn")
else:
    run("git", "clone", "--quiet", REPO_URL, str(REPO_DIR))
    print("đã clone mới")

%cd /kaggle/working/vihallulens
print("commit:", run("git", "log", "--oneline", "-1", cwd=REPO_DIR))

In [ ]:
# Ô 2 — cài đặt. bitsandbytes là thứ T18 không cần: nó dùng cho lượng tử hóa NF4 của mô
# hình đọc 7B. Không có nó thì mô hình phải nạp ở float16 và tràn 16 GB.
!pip install -q --no-deps -e .
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
# Ô 3 — chuẩn bị dữ liệu và kiểm tra môi trường. Khoảng 1 phút, chạy CPU.
# Bộ kiểm thử chạy trước khi tốn GPU: nó bắt được lỗi công thức mà không cần card.
get_ipython().system("python scripts/probe_env.py")
get_ipython().system("python scripts/normalize_data.py --dataset vihallu")
get_ipython().system("python scripts/split_data.py --only vihallu")
get_ipython().system("python -m pytest tests/test_lookback.py tests/test_splits.py -q")

## Trích đặc trưng

Hai ô, tập train trước vì nó dài hơn. Ô 6 chỉ mất vài giây và chạy trên CPU.

**Đọc gì trong lúc chạy:** dòng tiến độ in ra `ms/mẫu` và ước tính thời gian còn lại. Nếu
`ms/mẫu` vượt xa 420 thì có gì đó không như dự tính — nhưng cứ để chạy tiếp, vì phần đã tính
không mất đi đâu.

Cột `lỗi` phải là 0. Có lỗi thì mẫu đó bị bỏ qua chứ không làm dừng lượt chạy, và lỗi **đầu
tiên** được in nguyên văn để lần.

In [ ]:
# Ô 4 — trích đặc trưng tập train. Khoảng 39 phút. Chạy lại được, sẽ bỏ qua phần đã xong.
!python scripts/extract_features.py --config configs/e02_lookback_vihallu.yaml --split train

In [ ]:
# Ô 5 — trích đặc trưng tập test. Khoảng 5 phút.
!python scripts/extract_features.py --config configs/e02_lookback_vihallu.yaml --split test

In [ ]:
# Ô 6 — huấn luyện và chấm điểm. Vài giây, chạy CPU. Copy toàn bộ output.
!python scripts/run_lookback_baseline.py --config configs/e02_lookback_vihallu.yaml

In [ ]:
# Ô 7 — lấy kết quả về. Đặc trưng thô cũng tải xuống để lần sau khỏi chạy lại GPU:
# T21 tới T23 dùng chung cách trích này, và 50 phút quota đáng để giữ.
import shutil
from pathlib import Path

shutil.copy("results/runs.jsonl", "/kaggle/working/runs.jsonl")
for path in sorted(Path("data/processed").glob("*.jsonl")):
    size = path.stat().st_size / 1024**2
    shutil.copy(path, f"/kaggle/working/{path.name}")
    print(f"{path.name}  {size:.1f} MB")

with open("results/runs.jsonl", encoding="utf-8") as handle:
    print(handle.read())

## Đọc kết quả thế nào

**Tiêu chí hoàn thành của T20: macro-F1 phải cao hơn 0,6562 của baseline tầm thường E01.**
Script tự kiểm và tự nói đạt hay không đạt ở cuối.

Không đạt thì theo đúng T20 trong `TASKS.md`: **dừng lại rà soát khâu trích đặc trưng, đừng báo
cáo con số đó.** Lỗi gần như chắc chắn nằm ở khâu trích chứ không ở phương pháp, và script in
sẵn ba chỗ phải soi trước tiên.

Bốn thứ cần nhìn, theo thứ tự:

1. **Khối kiểm tra đặc trưng**, in trước cả điểm số. Tỷ lệ lookback theo định nghĩa phải nằm
   trong đoạn `[0, 1]`; nếu phần trăm nằm trong đoạn không xấp xỉ 100 thì công thức sai chứ
   không phải mô hình kém. Số đặc trưng không đổi mà lớn thì có lớp hoặc đầu chú ý bị chết.

2. **macro-F1 kèm khoảng tin cậy.** Mốc để nói *hơn hẳn* E01 là **0,689**, cận trên khoảng tin
   cậy của E01 — vượt 0,67 chỉ là nằm trong nhiễu của cùng một kết quả.

3. **Mười đầu chú ý được dựa vào nhiều nhất.** Bài gốc thấy tín hiệu dồn vào một số ít đầu.
   Trọng số trải đều khắp 756 đặc trưng là dấu hiệu đã tái lập **nhiễu** chứ không phải tín
   hiệu — điểm số có thể vẫn đẹp mà kết luận thì sai.

4. **Tỷ lệ bị cắt ngữ cảnh.** Đo ở T05: trần 4.096 token chỉ cắt rất ít mẫu ViHallu, nên con số
   này phải gần 0. Lớn bất thường nghĩa là chỗ dò vùng ngữ cảnh có vấn đề.

## Hai khác biệt so với bài gốc, phải nêu mỗi khi nêu con số

- Bài gốc lấy span bằng **cửa sổ trượt 8 token**; ViHallu gán nhãn cho cả phản hồi nên ở đây một
  span là **toàn bộ phản hồi**.
- Bài gốc phân loại **nhị phân** và báo **AUROC**; đây **ba lớp** và **macro-F1**. Hai con số
  không đặt cạnh nhau được.